In [48]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import sklearn
import numpy as np


In [49]:
# read the train set
trainset=pd.read_csv("house-prices-advanced-regression-techniques/train.csv")
# read the test set
testset=pd.read_csv("house-prices-advanced-regression-techniques/test.csv")
#trainset.head(100)
trainset.shape

(1460, 81)

<h>Filling null values</h1>


In [50]:
# sort it with non null value counts
print(trainset.count().sort_values(ascending=True).to_string())

PoolQC              7
MiscFeature        54
Alley              91
Fence             281
MasVnrType        588
FireplaceQu       770
LotFrontage      1201
GarageType       1379
GarageCond       1379
GarageFinish     1379
GarageYrBlt      1379
GarageQual       1379
BsmtExposure     1422
BsmtFinType2     1422
BsmtFinType1     1423
BsmtCond         1423
BsmtQual         1423
MasVnrArea       1452
Electrical       1459
Neighborhood     1460
BldgType         1460
Condition2       1460
Condition1       1460
Id               1460
LandContour      1460
Utilities        1460
LotConfig        1460
LandSlope        1460
HouseStyle       1460
OverallQual      1460
OverallCond      1460
MSSubClass       1460
ExterCond        1460
Foundation       1460
Exterior2nd      1460
ExterQual        1460
YearRemodAdd     1460
RoofStyle        1460
RoofMatl         1460
Exterior1st      1460
YearBuilt        1460
Heating          1460
CentralAir       1460
HeatingQC        1460
BsmtFinSF2       1460
BsmtUnfSF 

checking the features with most nulls, if null has a meaning (spoiler: it does here)

In [51]:
trainset["PoolQC"].unique()
# nan means no pool, which shouldnt be dropped 
# since its important for price
trainset.fillna({"PoolQC":"No Pool"}, inplace=True)

trainset["Fence"].unique()
#nan means no fence, which shouldnt be dropped 
# since it's important for price
trainset.fillna({"Fence":"No Fence"}, inplace=True)

trainset["MiscFeature"].unique()
trainset.fillna({"MiscFeature":"None"}, inplace=True)

trainset["Alley"].unique()
trainset.fillna({"Alley":"No Alley"}, inplace=True)

trainset["FireplaceQu"].unique()
trainset.fillna({"FireplaceQu":"No Fireplace"}, inplace=True)

trainset["MasVnrType"].unique()
trainset.fillna({"MasVnrType":"None"}, inplace=True)



for col in trainset.columns:
    if trainset[col].mode()[0] == np.nan:
        trainset.drop(col, axis=1, inplace=True)


first, lets try simple devision of training set and validation set 80:20

In [52]:
from sklearn.model_selection import train_test_split
trainsetLR=trainset.copy()
X = trainsetLR.drop("SalePrice", axis=1)
y = trainsetLR["SalePrice"]

X_train_LR, X_val_LR, y_train_LR, y_val_LR = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
X_train_LR["SalePrice"] = y_train_LR



first lets convert every categorical feature into numerical for linear regression, simply, lets assign each category to the mean of the target when grouped by that category

In [53]:

cat_cols = X_train_LR.select_dtypes(include="object").columns

for col in cat_cols:
    X_train_LR=X_train_LR.fillna(X_train_LR[col].mode()[0])
    X_val_LR=X_val_LR.fillna(X_val_LR[col].mode()[0])

cat_to_num_maps = {}

for col in cat_cols:
    cat_to_num_maps[col] = X_train_LR.groupby(col)["SalePrice"].mean()


for col in cat_cols:
    X_train_LR[col + "_num"] = X_train_LR[col].map(cat_to_num_maps[col])
    X_train_LR.drop(col, axis=1, inplace=True)
    X_val_LR[col + "_num"] = X_val_LR[col].map(cat_to_num_maps[col])
    X_val_LR.drop(col, axis=1, inplace=True)


X_train_LR.drop(columns=["Id", "SalePrice"], inplace=True)

X_train_LR.head()
#theres still some weird values in numerical, we can fill them with the mean of the column
X_train_LR = X_train_LR.replace("RL", np.nan)
X_val_LR = X_val_LR.replace("RL", np.nan)
X_train_LR = X_train_LR.replace("RM", np.nan)
X_val_LR = X_val_LR.replace("RM", np.nan)
X_train_LR = X_train_LR.replace("FV", np.nan)
X_val_LR = X_val_LR.replace("FV", np.nan)
X_train_LR = X_train_LR.replace("RP", np.nan)
X_val_LR = X_val_LR.replace("RP", np.nan)
X_train_LR = X_train_LR.replace("RH", np.nan)
X_val_LR = X_val_LR.replace("RH", np.nan)
X_train_LR = X_train_LR.replace("A", np.nan)
X_val_LR = X_val_LR.replace("A", np.nan)
X_train_LR = X_train_LR.replace("C", np.nan)
X_val_LR = X_val_LR.replace("C", np.nan)
X_train_LR = X_train_LR.replace("I", np.nan)
X_val_LR = X_val_LR.replace("I", np.nan)

for col in X_train_LR.columns:
    X_train_LR[col] = X_train_LR[col].fillna(X_train_LR[col].mean())
    X_val_LR[col] = X_val_LR[col].fillna(X_train_LR[col].mean())


/tmp/ipykernel_74906/122816007.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_LR = X_train_LR.replace("RL", np.nan)
/tmp/ipykernel_74906/122816007.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_val_LR = X_val_LR.replace("RL", np.nan)


In [54]:

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

scaler=StandardScaler()
scaled = scaler.fit_transform(X_train_LR)
scaledV = scaler.fit_transform(X_val_LR)


X_train_LR_scaled = pd.DataFrame(
    scaled,
    columns=X_train_LR.columns,
    index=X_train_LR.index
)
X_val_LR_scaled = pd.DataFrame(
    scaledV,
    columns=X_val_LR.columns,
    index=X_val_LR.index
)


modelLR = LinearRegression()
rfe=RFE(modelLR, n_features_to_select=20, step=1)
rfe.fit(X_train_LR_scaled, y_train_LR)
rfe_selected_features = X_train_LR_scaled.columns[rfe.support_].to_list()

for i, feature in enumerate(rfe_selected_features, 1):
    print(f"{i}. {feature}")



X_train_LR_scaled = X_train_LR_scaled[rfe_selected_features]
X_val_LR_scaled = X_val_LR_scaled[rfe_selected_features]


1. MSSubClass
2. LotFrontage
3. OverallQual
4. OverallCond
5. BsmtFinSF1
6. 1stFlrSF
7. 2ndFlrSF
8. GrLivArea
9. BsmtFullBath
10. TotRmsAbvGrd
11. Fireplaces
12. GarageCars
13. PoolArea
14. LandContour_num
15. Neighborhood_num
16. BsmtQual_num
17. BsmtExposure_num
18. KitchenQual_num
19. PoolQC_num
20. SaleCondition_num


X_train_LR is ready, now lets do mlflow log

In [55]:
#pip install mlflow

In [59]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error
mlflow.set_tracking_uri(
    "sqlite:////home/sandro/slomi23_ML_Assignment_1/mlflow.db"
)
mlflow.set_experiment("House Price Prediction")
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

pipeline.fit(X_train_LR, y_train_LR)
scalers = [
    StandardScaler(),
    MinMaxScaler(),
    None
]

param_grid = {
    'scaler': scalers,
    'classifier__C': [0.1, 1, 10],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='f1',
    verbose=2,
    return_train_score=True
)


with mlflow.start_run(run_name="linear_regression_without_WOE_and_kfold"):

    model = LinearRegression()
    model.fit(X_train_LR_scaled, y_train_LR)

    # predictions
    train_preds = model.predict(X_train_LR_scaled)
    val_preds = model.predict(X_val_LR_scaled)

    # metrics
    train_rmse = np.sqrt(mean_squared_error(y_train_LR, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val_LR, val_preds))

    # log metrics
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_params(model.get_params())
    print(f"Train RMSE: {train_rmse}")
    print(f"Val RMSE: {val_rmse}")

    # log model
    mlflow.sklearn.log_model(model, "linear_regression_model")


2026/04/10 21:54:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 21:54:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train RMSE: 29260.53981371497
Val RMSE: 33823.117012592


Train RMSE: 29260.53981371497
Val RMSE: 33823.117012592

now lets try kfold. still without WOE

In [ ]:
X_train_LR_KF=X_train_LR.copy().
